In [8]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!nvidia-smi
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import  sklearn
from sklearn.model_selection import train_test_split
#import ray
import os
import math
import matplotlib.pyplot as plt
import tensorflow.keras.optimizers.schedules as schedules
import pickle 

Thu Nov 16 01:12:34 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 520.61.05    Driver Version: 520.61.05    CUDA Version: 11.8     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA RTX A6000    On   | 00000000:1A:00.0 Off |                  Off |
| 30%   33C    P2    67W / 300W |    471MiB / 49140MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA RTX A6000    On   | 00000000:1B:00.0 Off |                  Off |
| 30%   

In [9]:
os.cpu_count()

32

In [10]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)


In [11]:
#ray.shutdown()
#ray.init(num_cpus=2, num_gpus=0)
#ray.shutdown()

# Data processing

For the homework, you should download the dataset with the provided link. The provided dataset has already partitioned the data by clients.

In [12]:
NUM_CLIENT = 100
train_data = 'train_data.npy'
train_data_arr = np.load(train_data, allow_pickle=True)
test_data = 'test_data.npy'
test_data_arr = np.load(test_data, allow_pickle=True)

# Define model

In [13]:
## regular model 
'''
def get_model():
    model = tf.keras.models.Sequential(
        [ tf.keras.layers.Flatten(input_shape=(28, 28)),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(62, activation="softmax")])
    model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
    return model
'''
## adding leanring rate.
def get_model(lr):
    lr_schedule = schedules.ExponentialDecay(
        initial_learning_rate=lr,
        decay_steps=10000,
        decay_rate=0.9,
        staircase=True
    )

    model = tf.keras.models.Sequential([
        tf.keras.layers.Flatten(input_shape=(28, 28)),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(62, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


'''
## normalized data 
def get_model():
    model = tf.keras.models.Sequential(
        [ tf.keras.layers.Lambda(lambda x: x / 255.0, input_shape=(28, 28)),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(62, activation="softmax")])
    model.compile("adam", "sparse_categorical_crossentropy", metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])
    return model
'''

'\n## normalized data \ndef get_model():\n    model = tf.keras.models.Sequential(\n        [ tf.keras.layers.Lambda(lambda x: x / 255.0, input_shape=(28, 28)),\n            tf.keras.layers.Flatten(),\n            tf.keras.layers.Dense(128, activation="relu"),\n            tf.keras.layers.Dense(128, activation="relu"),\n            tf.keras.layers.Dense(62, activation="softmax")])\n    model.compile("adam", "sparse_categorical_crossentropy", metrics=[tf.keras.metrics.SparseCategoricalAccuracy()])\n    return model\n'

## Splitting Data

In [14]:
all_train = []
all_val = []
x_train_all = []
y_train_all = []
x_val_all =[]
y_val_all =[]

for i in range(100):
    
    data = train_data_arr[i]
    # Split the data into training and validation sets
    images_train, images_val, labels_train, labels_val = train_test_split(
        data['images'],
        data['labels'],
        test_size=0.2,
        random_state=42)
    # Normalize the input data
    #images_train = np.array(images_train)
    #images_val = np.array(images_val)
    #images_train, images_val = images_train / 255.0, images_val / 255.0
    
    
    # Create TensorFlow datasets
    #expand dimensions of input to (1, 28, 28):
    #print("images before expansion: ", images_train.shape)
    #images_train = np.array([np.expand_dims(image, axis=0) for image in images_train])
    #images_val = [np.expand_dims(image, axis=0) for image in images_val]
    #images_train = np.expand_dims(images_train, axis=1)  # Add batch dimension
    #images_val = np.expand_dims(images_val, axis=1)
    #print("images expanded: ", images_train[0].shape)
    train_dataset = tf.data.Dataset.from_tensor_slices((images_train, labels_train))
    val_dataset = tf.data.Dataset.from_tensor_slices((images_val, labels_val))
    all_train.append(train_dataset)
    all_val.append(val_dataset)


# Client class

In [15]:
#@ray.remote(num_cpus=1, num_gpus=0)
class Client:
    def __init__(self, i, batch_size, initial_learning_rate, E):
        # Create model
        self.model = get_model(initial_learning_rate)
        self.training_data = all_train[i]
        self.validation_data = all_val[i]
        self.batch_size = batch_size
        self.E = E

    def train(self, global_weights):
        self.model.set_weights(global_weights)
        
        
        # Define a custom callback for printing validation metrics every 10 epochs
        '''
        class PrintValidationMetrics(tf.keras.callbacks.Callback):
            def on_epoch_end(self, epoch, logs=None):
                if epoch % 10 == 0:
                    val_loss, val_accuracy = self.model.evaluate(
                        self.validation_data.batch(self.batch_size), verbose=0)
                    print(f'Epoch {epoch}, Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')

        callback = PrintValidationMetrics()
        
        '''
        
        self.model.fit(self.training_data.batch(self.batch_size), epochs=self.E, validation_data=self.validation_data.batch(self.batch_size), verbose=0)
        ### to get accuracy and loss as arrays for all epochs: 
        '''
         history = self.model.fit(self.training_data.batch(self.batch_size), epochs=E, validation_data=self.validation_data.batch(self.batch_size), verbose=0)
        
        # Access the training history
        ## define objects of these arrays in constructor
        train_loss = history.history['loss']
        train_accuracy = history.history['accuracy']
        val_loss = history.history['val_loss']
        val_accuracy = history.history['val_accuracy']

        return train_loss, train_accuracy, val_loss, val_accuracy

        '''
        return self.model.get_weights()

    def evaluate(self):
        #gives scalars of loss and accuracy at the end of training 
        train_loss, train_accuracy = self.model.evaluate(self.training_data.batch(self.batch_size), verbose=0)
        val_loss, val_accuracy = self.model.evaluate(self.validation_data.batch(self.batch_size))
        return train_loss, train_accuracy, val_loss, val_accuracy


# Aggregated metrics and losses
Aggregate client metrics and losses for (federated) evaluation metrics.

In [16]:
sample_nbs = []
for i in range(100):
    #print(i)
    #print(len(train_data_arr[i]['labels']))
    #print("train data: ", len( all_train[i]))
    #print("80% of local data: ", len(train_data_arr[i]['labels'])*0.8)
    sample_nbs.append(len(train_data_arr[i]['labels']))
#sample_nbs

In [17]:
def weighted_average(sampled_clients, metric):
    # TODO: Aggregate client losses or accuracies by taking a weighted average.
    #       Weights are proportional to the number of samples at each client
    sample_size_selected = [sample_nbs[i] for i in sampled_clients]
    #sorted list so order should match
    agg_train_acc = sum(weight * loss for weight, loss in zip(sample_size_selected, metric['train_accuracy'])) / sum(sample_size_selected)
    agg_train_loss = sum(weight * loss for weight, loss in zip(sample_size_selected, metric['train_loss'])) / sum(sample_size_selected)
    agg_val_acc = sum(weight * loss for weight, loss in zip(sample_size_selected, metric['val_accuracy'])) / sum(sample_size_selected)
    agg_val_loss = sum(weight * loss for weight, loss in zip(sample_size_selected, metric['val_loss'])) / sum(sample_size_selected)


    return agg_train_acc, agg_train_loss, agg_val_acc, agg_val_loss

# Aggregation of client updates

In [18]:
def fedAvg(sampled_clients, client_weights):
    # Get the size of samples for each client
    sample_size_selected = [sample_nbs[c] for c in sampled_clients]
    #print("shape of local weights: ", len(client_weights[0]), client_weights[0][0].shape)
    list_dim = len(client_weights[0])
    np_dim = client_weights[0][0].shape
    total = sum(sample_size_selected)
    
    avg_weights = []
    for k in range(list_dim):
        avg_weights.append(np.zeros(client_weights[0][k].shape))

    for i in range (len(sampled_clients)):
        scalar = sample_size_selected[i]/total
        for j in range(0, list_dim):
            this_numpy = np.multiply(client_weights[i][j], scalar)
            avg_weights[j] = np.add(avg_weights[j],this_numpy)  
    #print("shape of avg weights: ", len(avg_weights), avg_weights[0].shape)
    return avg_weights


In [19]:
def plots_train(normalized, t, client, E, batch_size,  train_loss, train_accuracy, val_loss, val_accuracy):
    plt.figure(figsize=(12, 4))
    #plt.subplot(1, 2, 1)
    plt.plot(train_loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Rounds')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(normalized + '_train_loss_'+str(t)+'_'+str(client)+'_'+str(E)+'_'+str(batch_size)+'_.png')
    plt.close()
    # Plot training accuracy and validation accuracy
    plt.figure(figsize=(12, 4))
    plt.plot(train_accuracy, label='Training Accuracy')
    plt.plot(val_accuracy, label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Rounds')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.savefig(normalized + '_train_accuracy_'+str(t)+'_'+str(client)+'_'+str(E)+'_'+str(batch_size)+'_.png')
    plt.close()

# Launch the learning

In [20]:
run = "p1-C=8, E=20"
Agg_metric = {}
Agg_metric['train_loss'] = []
Agg_metric['train_accuracy'] =[]
Agg_metric['val_loss'] =[]
Agg_metric['val_accuracy'] =[]
    
# Initialize the global model weights
batch_size = 32
# Initialize the global model weights with an initial learning rate
initial_learning_rate = 0.001
global_model = get_model(initial_learning_rate)

#global_model = get_model()
users = np.arange(100)
# Get data
#data_by_clients, test_data = partition_data()

total_rounds = 200

C = 8   # Only a fraction of C of clients participate in the local training per round
E = 20     # Number of epochs for each local training

for round in range(total_rounds):
    # Step 1
    print("communication round: ", round)
    #initiate empty list of model weights 
    client_weights =[]
    metric = {}
    metric['train_loss'] = []
    metric['train_accuracy'] =[]
    metric['val_loss'] =[]
    metric['val_accuracy'] =[] 
    # TODO: Sample C fraction of clients
    sampled_clients = np.random.choice(users, C, replace=False)
    sampled_clients = sorted(sampled_clients)
    # TODO: Get the local data for the selected clients
    for i in sampled_clients:
        print("client: ", i)
        #perform local training
        #define instance of client:
        current_client = Client(i, batch_size, initial_learning_rate, E) 
        
        #save local weights
        
        local_weights = current_client.train(global_model.get_weights())
        #print(local_weights[0].shape)
        client_weights.append(local_weights)
        #save local results
        
        train_loss, train_accuracy, val_loss, val_accuracy = current_client.evaluate()
        metric['train_loss'].append(train_loss)
        metric['train_accuracy'].append(train_accuracy)
        metric['val_loss'].append(val_loss)
        metric['val_accuracy'].append(val_accuracy)
        #print("train_loss : ", train_loss)
        #print("train_accuracy: ", train_accuracy)
        

    # Step 3
    # TODO: Aggregate the losses and metrics, keep track of the metrics
    agg_train_acc, agg_train_loss, agg_val_acc, agg_val_loss = weighted_average(sampled_clients, metric)
    Agg_metric['train_loss'].append(agg_train_loss)
    Agg_metric['train_accuracy'].append(agg_train_acc)
    Agg_metric['val_loss'].append(agg_val_loss)
    Agg_metric['val_accuracy'].append(agg_val_acc)
    print("aggregated validation accuracy: ",Agg_metric['val_accuracy'][round]) 
    # Step 4
    # TODO: Aggregate the client updates
    weighted_avg_weights = fedAvg(sampled_clients, client_weights)
    
    # Step 5
    # Update the global model
    global_model.set_weights(weighted_avg_weights)
    
    if round%50==0 and round!=0:


    # Specify the file path
        file_path = 'Agg_metric_run3_'+str(round)+'_'+str(E)+'_'+str(C)+'_'+str(batch_size)+'_.pickle'

        # Save the dictionary to a pickle file
        with open(file_path, 'wb') as file:
            pickle.dump(Agg_metric, file)

        print(f'Dictionary saved to {file_path}')

        global_model.save('global_model_'+str(round)+'_'+str(E)+'_'+str(C)+'_'+str(batch_size)+'_.h5')
        plots_train(run, round, C, E, batch_size,  Agg_metric['train_loss'] , Agg_metric['train_accuracy'], Agg_metric['val_loss'], Agg_metric['val_accuracy'])
        

communication round:  0
client:  4


2023-11-16 01:12:52.740293: I tensorflow/stream_executor/cuda/cuda_blas.cc:1614] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


3/3 [==============================] - 0s 6ms/step - loss: 3.1652 - accuracy: 0.3421
client:  30
2/2 [==============================] - 0s 8ms/step - loss: 3.5261 - accuracy: 0.2857
client:  37
2/2 [==============================] - 0s 10ms/step - loss: 4.0027 - accuracy: 0.1875
client:  42
3/3 [==============================] - 0s 6ms/step - loss: 3.3443 - accuracy: 0.1892
client:  53
3/3 [==============================] - 0s 6ms/step - loss: 3.0712 - accuracy: 0.3378
client:  72
2/2 [==============================] - 0s 5ms/step - loss: 2.7162 - accuracy: 0.3750
client:  80
3/3 [==============================] - 0s 6ms/step - loss: 3.1188 - accuracy: 0.3099
client:  92
3/3 [==============================] - 0s 6ms/step - loss: 2.8264 - accuracy: 0.4267
aggregated validation accuracy:  0.3075217161075268
communication round:  1
client:  15
3/3 [==============================] - 0s 5ms/step - loss: 3.3959 - accuracy: 0.1912
client:  19
2/2 [==============================] - 0s 6ms/step

2023-11-16 01:15:22.195451: W tensorflow/core/data/root_dataset.cc:266] Optimization loop failed: CANCELLED: Operation was cancelled


3/3 [==============================] - 0s 9ms/step - loss: 2.5419 - accuracy: 0.4638
client:  63
3/3 [==============================] - 0s 6ms/step - loss: 2.1236 - accuracy: 0.5972
client:  69
2/2 [==============================] - 0s 8ms/step - loss: 2.3275 - accuracy: 0.5455
client:  92
3/3 [==============================] - 0s 12ms/step - loss: 1.7447 - accuracy: 0.6800
aggregated validation accuracy:  0.5890754985323132
communication round:  7
client:  18
3/3 [==============================] - 0s 7ms/step - loss: 2.3872 - accuracy: 0.5068
client:  24
2/2 [==============================] - 0s 11ms/step - loss: 2.3785 - accuracy: 0.5000
client:  34
3/3 [==============================] - 0s 6ms/step - loss: 1.9050 - accuracy: 0.6377
client:  36
2/2 [==============================] - 0s 12ms/step - loss: 2.0837 - accuracy: 0.4130
client:  47
3/3 [==============================] - 0s 6ms/step - loss: 2.1141 - accuracy: 0.6269
client:  48
2/2 [==============================] - 0s 11ms/s

...

2/2 [==============================] - 0s 9ms/step - loss: 2.8189 - accuracy: 0.5179
client:  90
3/3 [==============================] - 0s 4ms/step - loss: 1.3071 - accuracy: 0.6479
client:  94
3/3 [==============================] - 0s 5ms/step - loss: 1.6659 - accuracy: 0.7051
aggregated validation accuracy:  0.6728572798482245
communication round:  172
client:  3
3/3 [==============================] - 0s 4ms/step - loss: 1.4464 - accuracy: 0.7536
client:  4
3/3 [==============================] - 0s 5ms/step - loss: 1.0287 - accuracy: 0.8026
client:  12
2/2 [==============================] - 0s 5ms/step - loss: 2.2112 - accuracy: 0.6538
client:  44
2/2 [==============================] - 0s 7ms/step - loss: 1.7613 - accuracy: 0.6964
client:  63
3/3 [==============================] - 0s 4ms/step - loss: 1.6208 - accuracy: 0.6806
client:  82
3/3 [==============================] - 0s 7ms/step - loss: 1.5963 - accuracy: 0.6866
client:  96
2/2 [==============================] - 0s 7ms/step 

2023-11-16 02:31:40.776824: W tensorflow/core/data/root_dataset.cc:266] Optimization loop failed: CANCELLED: Operation was cancelled


2/2 [==============================] - 0s 6ms/step - loss: 2.2959 - accuracy: 0.6429
client:  70
3/3 [==============================] - 0s 6ms/step - loss: 2.4390 - accuracy: 0.6716
client:  89
3/3 [==============================] - 0s 6ms/step - loss: 1.6148 - accuracy: 0.6716
aggregated validation accuracy:  0.6338352046869256
communication round:  189
client:  1
3/3 [==============================] - 0s 5ms/step - loss: 1.4109 - accuracy: 0.7333
client:  6
3/3 [==============================] - 0s 5ms/step - loss: 0.9105 - accuracy: 0.8529
client:  36
2/2 [==============================] - 0s 6ms/step - loss: 1.9814 - accuracy: 0.5000
client:  49
2/2 [==============================] - 0s 6ms/step - loss: 2.2857 - accuracy: 0.6500
client:  54
3/3 [==============================] - 0s 3ms/step - loss: 1.4323 - accuracy: 0.6944
client:  75
2/2 [==============================] - 0s 7ms/step - loss: 1.4804 - accuracy: 0.6182
client:  77
2/2 [==============================] - 0s 8ms/step 

2023-11-16 02:33:47.175148: W tensorflow/core/data/root_dataset.cc:266] Optimization loop failed: CANCELLED: Operation was cancelled


2/2 [==============================] - 0s 6ms/step - loss: 3.2074 - accuracy: 0.7750
client:  49
2/2 [==============================] - 0s 13ms/step - loss: 2.0583 - accuracy: 0.6333
client:  50
3/3 [==============================] - 0s 6ms/step - loss: 1.4257 - accuracy: 0.7015
client:  57
3/3 [==============================] - 0s 7ms/step - loss: 0.5819 - accuracy: 0.8667
client:  84
2/2 [==============================] - 0s 6ms/step - loss: 2.7479 - accuracy: 0.5179
aggregated validation accuracy:  0.6901485542770431
communication round:  195
client:  2
2/2 [==============================] - 0s 6ms/step - loss: 2.4129 - accuracy: 0.6562
client:  18
3/3 [==============================] - 0s 7ms/step - loss: 2.1207 - accuracy: 0.6164
client:  22
3/3 [==============================] - 0s 5ms/step - loss: 1.6320 - accuracy: 0.5692
client:  42
3/3 [==============================] - 0s 4ms/step - loss: 1.3362 - accuracy: 0.7432
client:  63
3/3 [==============================] - 0s 8ms/ste

In [21]:
##loading saved model
'''
# Load the model from the saved file
loaded_model = tf.keras.models.load_model('my_model.h5')

# Now you can use loaded_model for predictions or further training
import tensorflow as tf

# Load the saved model
loaded_model = tf.keras.models.load_model('my_model.h5')

# Get additional training data
additional_data = ...

# Compile the model if needed (only if you did not save the optimizer state)
loaded_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Continue training with additional data
loaded_model.fit(additional_data, epochs=additional_epochs)

# Save the model with updated weights
loaded_model.save('my_model_updated.h5')

print('Updated model saved to my_model_updated.h5')


'''

"\n# Load the model from the saved file\nloaded_model = tf.keras.models.load_model('my_model.h5')\n\n# Now you can use loaded_model for predictions or further training\nimport tensorflow as tf\n\n# Load the saved model\nloaded_model = tf.keras.models.load_model('my_model.h5')\n\n# Get additional training data\nadditional_data = ...\n\n# Compile the model if needed (only if you did not save the optimizer state)\nloaded_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])\n\n# Continue training with additional data\nloaded_model.fit(additional_data, epochs=additional_epochs)\n\n# Save the model with updated weights\nloaded_model.save('my_model_updated.h5')\n\nprint('Updated model saved to my_model_updated.h5')\n\n\n"

# Create plots

In [28]:
def test(test_data_arr, weights, batch_size, lr):
    model = get_model(lr)  # Construct the model
    model.set_weights(weights)  # Set model weights with the latest parameters

    # Assuming test_data is a tuple of test_images and test_labels
    test_images = np.array(test_data_arr[0]['images'])
    test_labels = np.array(test_data_arr[0]['labels'])
    
 

    # Normalize the input data
    #test_images = np.array(test_images)
    #test_images= test_images / 255.0
    
    test_images_flat = test_images.reshape(test_images.shape[0], -1)
    
    test_dataset = tf.data.Dataset.from_tensor_slices((test_images, test_labels))
    batch_size = batch_size  # Set your desired batch size
    # Evaluate the model using the dataset
    batched_test_dataset = test_dataset.batch(batch_size)

    test_loss, test_accuracy = model.evaluate(batched_test_dataset)


    print(f"Test Accuracy: {test_accuracy:.4f}")

    return test_loss, test_accuracy

In [29]:
plots_train(run, total_rounds, C, E, batch_size, Agg_metric['train_loss'] , Agg_metric['train_accuracy'], Agg_metric['val_loss'], Agg_metric['val_accuracy'])
# Evaluate on test data after training
test_loss, test_accuracy = test(test_data_arr, global_model.get_weights(), batch_size, initial_learning_rate)

114/114 [==============================] - 1s 2ms/step - loss: 1.9016 - accuracy: 0.6023
Test Accuracy: 0.6023


In [30]:
print("test loss: ", test_loss)
print("test accuracy: ", test_accuracy)

test loss:  1.9015841484069824
test accuracy:  0.6023197770118713


In [31]:
test_results = np.array ([test_loss, test_accuracy])
np.save('test_results_'+run+'_'+str(total_rounds)+'_'+str(C)+'_'+str(E)+'_'+str(batch_size)+'_.npy', test_results)

In [32]:
##load pickle file 
'''
# Load the dictionary from the pickle file
with open(file_path, 'rb') as file:
    loaded_dict = pickle.load(file)

print('Loaded dictionary:', loaded_dict)
'''

"\n# Load the dictionary from the pickle file\nwith open(file_path, 'rb') as file:\n    loaded_dict = pickle.load(file)\n\nprint('Loaded dictionary:', loaded_dict)\n"

In [33]:
print("last train acc: ", Agg_metric['train_accuracy'][199])
print("last validation acc: ", Agg_metric['val_accuracy'][199])

last train acc:  0.8337148912151731
last validation acc:  0.712373536827449
